# Audience-pair pilot and pituitary comparison
Run on a GPU with access to HF_TOKEN and the Drive account containing your original ear-pressure pilot. This experiment starts from the same base weights and uses the original two examples plus three new General/Clinician pairs. One optimizer update accumulates all eight examples. Each run has its own output folder. The examples are illustrative and have no independent clinical review.

The final table compares base Llama, the original adapter and the new adapter on the same fixed pituitary question and excerpts. Since you have already seen this case, treat it as a development check. Compare directness, source attribution and appropriate vocabulary for each audience.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, datetime, uuid
SOURCE_COMMIT = '6b25fe2a11ce856746cc55bd7173dccce1e71933'
REPO = Path('/content') / ('healthbrowse-pilot-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--filter=blob:none', '--branch', 'research/healthbrowse-medical-model',
                    'https://github.com/abby2395/medical_chatbot.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError('The pilot checkout does not match the pinned source commit.')
LAB = REPO / 'healthbrowse_model_lab'
os.chdir(LAB)
print('Pinned pilot source:', head)


In [ ]:
# Keep the text-model stack independent of packages preinstalled in the Colab kernel.
import venv
ENV_DIR = Path('/content') / ('healthbrowse-train-' + sys.implementation.cache_tag)
venv.EnvBuilder(with_pip=False).create(ENV_DIR)
LAB_PYTHON = str(ENV_DIR / 'bin/python')
lab_env = os.environ.copy()
lab_env.pop('PYTHONPATH', None)
lab_env['PYTHONNOUSERSITE'] = '1'

def run_lab(*args):
    subprocess.run([LAB_PYTHON, *map(str, args)], cwd=LAB, env=lab_env, check=True)

# Target the venv directly; Colab's system Python may not ship ensurepip.
subprocess.run([sys.executable, '-m', 'pip', '--python', LAB_PYTHON,
                'install', '-e', '.[train,test]'], cwd=LAB, env=lab_env, check=True)
run_lab('-c', """import torch, transformers, accelerate, bitsandbytes
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.integrations import get_reporting_integration_callbacks
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('Trainer imports passed.', 'torch', torch.__version__, 'transformers', transformers.__version__)
""")


In [ ]:
from google.colab import userdata
run_lab('-m', 'hb_model_lab.cli', 'preflight')
run_lab('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime before training.'")
try:
    lab_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError("Enable notebook access to HF_TOKEN in Colab Secrets.") from exc


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base_dir = Path('/content/drive/MyDrive/HealthBrowseModelLab')
runs = sorted(p for p in base_dir.glob('ear-pressure-*') if (p / 'model/adapter/adapter_model.safetensors').is_file())
if not runs:
    raise FileNotFoundError(f'No completed ear-pressure run found in {base_dir}. Mount the Drive account used for training.')
if len(runs) == 1:
    run = runs[0]
else:
    print('Completed runs:', ', '.join(p.name for p in runs))
    selected = input('Enter the exact folder name: ').strip()
    run = base_dir / selected
    if run not in runs:
        raise ValueError('Select a folder from the list.')

first = run
run = base_dir / ('audience-pairs-' + uuid.uuid4().hex[:12])
report = run / 'pituitary-comparison.json'
print('Original adapter:', first)
print('New run:', run)


In [ ]:
prepare = r"""
import gzip
import json
import sys
from pathlib import Path
from hb_model_lab.data import digest, read_jsonl, validate_rows, validate_splits
from hb_model_lab.runtime import adapter_manifest
from hb_model_lab.training import load_config

first, run = map(Path, sys.argv[1:])
config = load_config(first / 'pilot-config.json')
manifest = adapter_manifest(first / 'model/adapter', config, allow_smoke=True)
train = first / 'data/train.jsonl'
val = first / 'data/validation.jsonl'
if digest(train) != manifest['train_sha256'] or digest(val) != manifest['validation_sha256']:
    raise ValueError('Original pilot data hashes differ from its manifest.')
rows = read_jsonl(train)
if len(rows) != 2 or {r['topic_group'] for r in rows} != {'middle-ear-pressure'}:
    raise ValueError('Select the original two-example pilot.')
rows += read_jsonl('data/examples/audience-pairs.jsonl')
if len(rows) != 8:
    raise ValueError('Expected four pairs.')
for group in {r['topic_group'] for r in rows}:
    assert sorted(r['input']['role'] for r in rows if r['topic_group'] == group) == ['clinician','general']
validation = read_jsonl(val)
validate_rows(rows, allow_examples=True)
validate_rows(validation, allow_examples=True)
validate_splits(rows, validation)
with gzip.open('data/education-corpus.jsonl.gz','rt',encoding='utf-8') as f:
    corpus = {p['id']:p for p in map(json.loads,f)}
held_out = corpus['m46699-fs-id2955108']['url']
for row in rows + validation:
    for e in row['input']['evidence']:
        assert e['url'] != held_out and 'pituitar' not in json.dumps(row).lower()
        assert e['text'] in corpus[e['id']]['text'] and e['url'] == corpus[e['id']]['url']
config.update(batch_size=1, gradient_accumulation_steps=len(rows))
run.mkdir(exist_ok=False)
(run / 'data').mkdir()
(run / 'data/train.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False) + '\n' for r in rows))
(run / 'data/validation.jsonl').write_bytes(val.read_bytes())
(run / 'pilot-config.json').write_text(json.dumps(config,indent=2))
print('Verified eight training examples, four validation examples and the pituitary topic exclusion.')
"""
run_lab("-c", prepare, first, run)


In [ ]:
run_lab('-m', 'hb_model_lab.cli', 'train', '--config', run / 'pilot-config.json',
        '--train', run / 'data/train.jsonl', '--validation', run / 'data/validation.jsonl',
        '--output', run / 'model', '--smoke')
print('New adapter:', run / 'model/adapter')


In [ ]:
experiment = r"""
import gzip
import json
import sys
from pathlib import Path

import torch
from peft import PeftModel
from hb_model_lab.contracts import Query, validate_answer
from hb_model_lab.data import digest, read_jsonl
from hb_model_lab.prompts import messages
from hb_model_lab.runtime import ModelRunner, adapter_manifest
from hb_model_lab.training import load_config

first, root, destination = map(Path, sys.argv[1:])
config = root / 'pilot-config.json'
adapter = root / 'model/adapter'
manifest = adapter_manifest(adapter, load_config(config), allow_smoke=True)
original = adapter_manifest(first / 'model/adapter', load_config(first / 'pilot-config.json'), allow_smoke=True)
question = 'What is the main function of the pituitary gland?'
ids = ['m46699-fs-id2955108', 'm46699-fs-id2161906']
with gzip.open('data/education-corpus.jsonl.gz', 'rt', encoding='utf-8') as f:
    passages = {r['id']: r for r in map(json.loads, f) if r['id'] in ids}
evidence = [{k: passages[i][k] for k in ('id','title','url','text')} for i in ids]
queries = [Query(question=question, role=r, mode='chat', topic_kind='knowledge',
                 evidence=evidence) for r in ('general', 'clinician')]
runner = ModelRunner(config, baseline=True)
tokenizer = runner.tokenizer
results = []
for condition in ('Base Llama', 'Original pilot', 'Expanded pilot'):
    if condition == 'Original pilot':
        runner.model = PeftModel.from_pretrained(runner.model, str(first / 'model/adapter'), is_trainable=False)
        runner.model.eval()
    elif condition == 'Expanded pilot':
        runner.model.load_adapter(str(adapter), adapter_name='expanded', is_trainable=False)
        runner.model.set_adapter('expanded')
        runner.model.eval()
    for query in queries:
        prompt = tokenizer.apply_chat_template(messages(query), tokenize=True,
                  add_generation_prompt=True, return_tensors='pt', return_dict=False).to(runner.model.device)
        with torch.inference_mode():
            output = runner.model.generate(prompt, attention_mask=torch.ones_like(prompt),
                      max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id,
                      eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')])
        raw = tokenizer.decode(output[0, prompt.shape[1]:], skip_special_tokens=True)
        record = dict(condition=condition, role=query.role, raw=raw)
        try:
            answer = validate_answer(json.loads(raw), query)
            record.update(status=answer.status, validated=answer.model_dump())
        except ValueError as exc:
            record.update(status='format or citation check failed', error=str(exc))
        results.append(record)
        print(condition, query.role, record['status'], flush=True)
with destination.open('x') as f:
    json.dump(dict(question=question, evidence=evidence, adapter_sha256=manifest['adapter_sha256'],
                   original_adapter_sha256=original['adapter_sha256'], max_new_tokens=512, results=results), f, indent=2, ensure_ascii=False)
"""
run_lab("-c", experiment, first, run, report)


In [ ]:
import html
from IPython.display import HTML, display
result = json.loads(report.read_text())
print('Question:', result['question'])
for e in result['evidence']:
    print(e['id'], e['title'], '\n' + e['text'] + '\n')
def show_record(r):
    if 'validated' not in r:
        return html.escape(r['status'] + '\n' + r['raw'])
    a = r['validated']
    lines = [claim['text'] for claim in a['answer']] or [a['status'], a['message'], *a['choices']]
    return html.escape('\n'.join(lines))
records = {(r['role'], r['condition']): r for r in result['results']}
table = '<table><tr><th>Audience</th><th>Base Llama</th><th>Original pilot</th><th>Expanded pilot</th></tr>'
for role in ['general','clinician']:
    table += '<tr><th>' + role.title() + '</th>'
    for model in ['Base Llama','Original pilot','Expanded pilot']:
        table += '<td style="white-space:pre-wrap;vertical-align:top">' + show_record(records[role,model]) + '</td>'
    table += '</tr>'
display(HTML(table + '</table>'))
print('Saved comparison:', report)
